# 04 - Despliegue y Puesta en Producci?n

## 1. Descripci?n General

Una vez finalizado el proceso de exploraci?n, preparaci?n de datos, ingenier?a de caracter?sticas, as? como el entrenamiento, evaluaci?n y selecci?n final del modelo, el siguiente paso consiste en preparar el sistema para su utilizaci?n en un entorno de producci?n.

En este notebook se construye y valida el flujo de inferencia que posteriormente ser? utilizado por la aplicaci?n. A diferencia del proceso de entrenamiento, donde el objetivo es optimizar y evaluar el modelo, aqu? el enfoque se centra en garantizar que un nuevo listing pueda recorrer correctamente todo el pipeline de producci?n: desde la preparaci?n de sus variables, la generaci?n de la predicci?n y el c?lculo de m?tricas complementarias, hasta la obtenci?n de una respuesta lista para ser consumida por una interfaz de usuario.

Este notebook no implementa la aplicaci?n final ni el servicio web. Su prop?sito es validar de manera aislada los distintos m?dulos de inferencia antes de integrarlos con FastAPI y Streamlit en la etapa de despliegue.

En particular, su alcance incluye:

- Verificar la carga correcta de los artefactos de producci?n.
- Validar el pipeline completo de preparaci?n de caracter?sticas para nuevos listings.
- Generar predicciones utilizando el modelo entrenado.
- Calcular m?tricas complementarias de mercado, como el precio t?pico, el rango de precios y la posici?n relativa del listing frente a alojamientos comparables.
- Comprobar el funcionamiento de los m?dulos que ser?n reutilizados durante el despliegue de la aplicaci?n.

Al finalizar este notebook, se contar? con un pipeline de inferencia completamente funcional y validado, listo para ser integrado en la API y la interfaz interactiva que conformar?n la aplicaci?n final.

## 2. Importaci?n de Librer?as y Carga del Dataset de Prueba

In [ ]:
# Import libraries and modules
import pandas as pd
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

# Load the autoreload extension to automatically reload modules when they change
# Set autoreload mode to 2: reload all modules before executing user code
%load_ext autoreload
%autoreload 2

# Import service modules for feature preparation, inference, and insights
from app.services.features import prepare_features
from app.services.inference import predict_listing
from app.services.insights import generate_market_insights

# Import feature list from model configuration
from src.settings.model_settings import MODEL_FEATURES

# Load the dataset generated before the Feature Engineering phase, to be used for testing the deployment pipeline.
df_test = pd.read_csv("../data/processed/df_features.csv")

# Preview first rows
df_test.head()

## 3. Prueba del Pipeline de Preprocesamiento e Inferencia

Antes de integrar el modelo en una API y una interfaz de usuario, es importante verificar que el pipeline de producci?n funcione correctamente de principio a fin. En esta secci?n se realiza una prueba utilizando un listing del conjunto de prueba, simulando el flujo que seguir? un nuevo alojamiento al utilizar la aplicaci?n.

Durante la inferencia, el listing atraviesa todas las etapas necesarias para generar una recomendaci?n de precio: preparaci?n de caracter?sticas, aplicaci?n del pipeline de preprocesamiento, predicci?n mediante el modelo entrenado y c?lculo de m?tricas complementarias de mercado. El objetivo no es volver a evaluar el desempe?o del modelo, sino validar que todos los componentes de producci?n interact?an correctamente y producen una salida coherente.

In [ ]:
# END-TO-END PREPROCESSING & INFERENCE TEST

# Select one listing from the test set
listing = df_test.iloc[[0]].copy()

print()
print("RAW LISTING:")
print(f"Shape: {listing.shape}")
display(listing.head())

# Prepare features
prepared_listing = prepare_features(listing)

print()
print("PREPARED FEATURES:")
print(f"Shape: {prepared_listing.shape}")
display(prepared_listing.head())

# Generate prediction
prediction = predict_listing(listing)

# Generate market insights
insights = generate_market_insights(
    listing=listing,
    predicted_price=prediction
)

# Display results
print()
print("=" * 60)
print("SMART PRICING RESULTS")
print("=" * 60)

print(f"Estimated Price      : ${prediction:,.2f} MXN")
print(f"Typical Market Price : ${insights['market_median']:,.2f} MXN")
print(
    f"Market Price Range   : "
    f"${insights['market_price_min']:,.2f}"
    f" - "
    f"${insights['market_price_max']:,.2f} MXN"
)
print(f"Listing Position     : {insights['listing_position']}")
print(f"Comparables listings : {insights['num_comparables']}")
print(f"Confidence           : {insights['confidence']}")
print(f"Search Radius (Km)   : {insights['search_radius_km']}")

La prueba confirma que el pipeline de inferencia funciona correctamente de extremo a extremo. A partir de un nuevo listing, el sistema aplica autom?ticamente la preparaci?n de caracter?sticas, el pipeline de preprocesamiento entrenado y el modelo de regresi?n (XGBoost) para generar una recomendaci?n de precio, sin requerir intervenci?n manual.

Para este alojamiento, el modelo estima un precio de \$1,180.08 MXN, valor que se encuentra dentro del rango de precios observado en el mercado para alojamientos comparables (\$948.25 ? \$1,622.75 MXN). Asimismo, el precio t?pico de estos comparables es de \$1,205.50 MXN, lo que indica que la recomendaci?n del modelo es consistente con el comportamiento del mercado local.

La b?squeda identific? 90 alojamientos comparables utilizando un radio de 2 km, proporcionando una base suficiente para calcular las m?tricas complementarias. Debido al n?mero de comparables encontrados, el sistema asign? un nivel de confianza High, sugiriendo que la estimaci?n se apoya en una muestra representativa de listings similares. En conjunto, estos resultados validan que el pipeline de producci?n no solo genera una predicci?n del precio, sino tambi?n informaci?n contextual que facilita la interpretaci?n de dicha recomendaci?n.


Con esta validaci?n concluye el desarrollo del pipeline de inferencia del proyecto. Se comprob? que un nuevo listing puede recorrer correctamente todas las etapas necesarias para generar una recomendaci?n de precio, desde la preparaci?n de caracter?sticas y el preprocesamiento hasta la predicci?n del modelo y el c?lculo de m?tricas complementarias de mercado. Con el pipeline de inferencia validado, el siguiente paso consiste en integrarlo dentro de una aplicaci?n interactiva que permita a cualquier usuario obtener recomendaciones de precio sin necesidad de ejecutar c?digo.

A partir de este punto, el proyecto est? listo para ser integrado en una aplicaci?n interactiva. La implementaci?n del sistema se realizar? utilizando **FastAPI** como capa de servicio para exponer el modelo mediante una API REST y **Streamlit** como interfaz de usuario para capturar las caracter?sticas de un alojamiento y presentar las recomendaciones de precio junto con los insights de mercado.


Las pruebas realizadas en este notebook concluyen la validaci?n del pipeline de producci?n y representan la ?ltima etapa del desarrollo del modelo dentro de este proyecto. A partir de este punto, el trabajo continuar? directamente en la implementaci?n de la aplicaci?n, donde el pipeline ser? integrado con FastAPI y Streamlit. Las validaciones restantes corresponder?n al correcto funcionamiento de la API, la comunicaci?n con la interfaz y la experiencia de usuario, sin requerir nuevas evaluaciones del modelo predictivo o nuevas validaciones en este notebook.